In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 84 samples and 13998 columns.
First few rows of the data:


/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_53349/579175482.py:28: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(preprocessed_csv_path)


,pfs.mos..,recurrence,os.mos..,vitalstatus,sample_title,stic,age,path.stage,surgical.outcome,diagnosis.of.record,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,19.0,1,34.0,1.0,HGSC_40190,Yes,61,III,Optimal,Not available,...,6.040067,5.035451,5.740524,6.091217,4.829373,5.089559,6.619170,6.902299,6.207234,6.313062
1,40.0,1,64.0,0.0,HGSC_40191,No,49,III,Optimal,Not available,...,6.537236,5.038692,5.921121,6.717286,5.075715,5.334657,6.859956,6.920267,6.185481,6.279690
2,42.0,1,72.0,0.0,HGSC_40193,No,53,II,Unknown,Not available,...,7.070010,6.296203,7.057889,6.890579,5.866339,6.279101,7.889931,8.161322,7.570871,7.458223
3,32.0,1,34.0,1.0,HGSC_40197,Yes,53,III,Optimal,Not available,...,6.976777,6.125367,6.876939,7.116488,5.835287,6.258743,7.824677,8.059266,7.367960,7.466249
4,13.0,1,39.0,1.0,HGSC_40199,No,49,III,Suboptimal,Not available,...,6.490908,5.032042,5.987931,6.960127,5.560689,5.339411,6.856786,7.024983,6.217915,6.551331


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for pfs.mos.. or recurrence: 0
Number of samples with complete data for both variables: 84
Number of censored samples (event_var == 0.0): 28
Number of events (event_var == 1.0): 56
Censored ratio (among complete): 28 / 84 = 0.333 (33.3%)
Number of samples with incomplete data for os.mos.. or vitalstatus: 1
Number of samples with complete data for both variables: 83
Number of censored samples (event_var == 0.0): 64
Number of events (event_var == 1.0): 19
Censored ratio (among complete): 64 / 83 = 0.771 (77.1%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE102073/description.json
